##Ingestão da tabela - Trips

In [0]:
from pyspark.sql import functions as F

##Configuração do caminho

In [0]:
# 1. Guarda o caminho do arquivo
# trips_path é o nome da variavel em Python
trips_path = "/Volumes/workspace/default/logistics_operations_raw/trips.csv" # A variável trips_path recebe o caminho do arquivo trips.csv

##Leitura do arquivo

In [0]:
# 2. Lê o CSV e cria o DataFrame
df_trips =(
    spark.read # Inicia a leitura
    .option("header", True) # Usa a primeira linha como nome das colunas
    .option("inferSchema", True) # Tenta indentificar tipos como texto, inteiro, decimal e data
    .csv(trips_path) # Lê o arquivo indicado
)

##Visualizar os dados

In [0]:
display(df_trips.limit(10))

##Tipos das colunas

In [0]:
df_trips.printSchema() # Mostra o tipo das colunas (string, double, int..)

In [0]:
total_linhas = df_trips.count() # Conta a quantidade de linhas que existem
total_viagens_unicas = df_trips.select("trip_id").distinct().count() # Seleciona o trip_id, o distinct() remove duplicatas, e o count conta quantos trip_id diferentes existem
print(f"Total de linhas: {total_linhas}") # Exibi o valor total de total_linhas. O f permite inserir uma variavel dentro do texto usando chaves 
print(f"Total de viagens únicas: {total_viagens_unicas}") # Exibi o valor total de total_linhas unicas encontradas. O f permite inserir uma variavel dentro do texto usando chaves 

In [0]:
df_trips.filter(df_trips.trip_id.isNull()).count() # Verifica se tem nulo

##Checagem de Nulos

In [0]:
df_trips.select([
    F.count(
        F.when(F.col(coluna).isNull(), 1)
    ).alias(coluna)
    for coluna in df_trips.columns
]).display()

##Investigar os registros

In [0]:
display(
    df_trips
    .filter(
        F.col("driver_id").isNull() |
        F.col("truck_id").isNull() |
        F.col("trailer_id").isNull()
    )
    .limit(20)
)

##Contar separadamente os nulos de cada recurso por status

In [0]:
display(
    df_trips
    .groupBy("trip_status")
    .agg(
        F.sum(F.when(F.col("driver_id").isNull(), 1).otherwise(0)).alias("motoristas_nulos"),
        F.sum(F.when(F.col("truck_id").isNull(), 1).otherwise(0)).alias("caminhoes_nulos"),
        F.sum(F.when(F.col("trailer_id").isNull(), 1).otherwise(0)).alias("carretas_nulas")
    )
    .orderBy("trip_status")
)

##Em qual status essas ausências acontecem

In [0]:
display(
    df_trips
    .filter(
        F.col("driver_id").isNull() |
        F.col("truck_id").isNull() |
        F.col("trailer_id").isNull()
    )
    .groupBy("trip_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_trips
    .filter(
        F.col("driver_id").isNull() |
        F.col("truck_id").isNull() |
        F.col("trailer_id").isNull()
    )
    .groupBy(
        F.col("driver_id").isNull().alias("driver_nulo"),
        F.col("truck_id").isNull().alias("truck_nulo"),
        F.col("trailer_id").isNull().alias("trailer_nulo")
    )
    .count()
    .orderBy(F.desc("count"))
)

##Salvar o DataFrame como tabela Delta

In [0]:
(
    df_trips.write # Inicia a gravação do DataFrame
    .format("delta") # Define o formato Delta Lake
    .mode("overwrite") # Substitui a tabela caso ela já exista
    .saveAsTable("workspace.bronze_logistics.trips") # Cria a tabela no catálogo
)

##Ingestão da tabela - Loads

##Configuração do caminho

In [0]:
loads_path = "/Volumes/workspace/default/logistics_operations_raw/loads.csv"

##Leitura do Arquivo

In [0]:
# 2. Lê o CSV e cria o DataFrame
df_loads =(
    spark.read # Inicia a leitura
    .option("header", True) # Usa a primeira linha como nome das colunas
    .option("inferSchema", True) # Tenta indentificar tipos como texto, inteiro, decimal e data
    .csv(loads_path) # Lê o arquivo indicado
)

##Visualização dos dados

In [0]:
display(df_loads.limit(10))

##Tipos das colunas

In [0]:
df_loads.printSchema() # Mostra o tipo das colunas (string, double, int..)

In [0]:
total_linhas_loads = df_loads.count() # Conta a quantidade de linhas que existem

total_loads_unicos = (
    df_loads.select("load_id")
    .distinct()# Seleciona o load_id, o distinct() remove duplicatas
    .count() # O count conta quantos load_id diferentes existem
)

load_id_unicos = (
    df_loads
    .filter(F.col("load_id").isNull())
    .count()
)
print(f"Total de linhas: {total_linhas_loads}") # Exibi o valor total de total_linhas. O f permite inserir uma variavel dentro do texto usando chaves 
print(f"Load únicos: {total_loads_unicos}") # Exibi o valor total de total_linhas unicas encontradas. O f permite inserir uma variavel dentro do texto usando chaves 
print(f"Load Ids únicos: {load_id_unicos}")

## Verificar nulos nas colunas

In [0]:
display(
    df_loads.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_loads.columns
    ])
)

##Conferir valores categóricos

In [0]:
display(
    df_loads
    .groupBy("load_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_loads
    .groupBy("load_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_loads
    .groupBy("booking_type")
    .count()
    .orderBy(F.desc("count"))
)

## Validar regras númericas básicas

In [0]:
display(
    df_loads.select(
        F.sum(F.when(F.col("weight_lbs") <= 0, 1).otherwise(0)).alias("peso_nao_positivo"),
        F.sum(F.when(F.col("pieces") <= 0, 1).otherwise(0)).alias("pecas_nao_positivas"),
        F.sum(F.when(F.col("revenue") < 0, 1).otherwise(0)).alias("receita_negativa"),
        F.sum(F.when(F.col("fuel_surcharge") < 0, 1).otherwise(0)).alias("frete_combustivel_negativo"),
        F.sum(F.when(F.col("accessorial_charges") < 0, 1).otherwise(0)).alias("encargos_negativos")
    )
)

In [0]:
loads_sem_trip = (
    df_loads
    .join(
        df_trips.select("load_id"),
        on="load_id",
        how="left_anti"
    )
)

print(f"Loads sem viagem correspondente: {loads_sem_trip.count()}")

##Verificar relacionamento com trips

In [0]:
loads_nao_encontrados = (
    df_trips
    .join(
        df_loads.select("load_id"),
        on="load_id",
        how="left_anti"
    )
)

print(f"Viagens com load_id não encontrado em loads: {loads_nao_encontrados.count()}")

##Gravar tabela Delta

In [0]:
(
    df_loads.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.loads")
)

In [0]:
spark.table("workspace.bronze_logistics.loads").count()

##Ingestão da tabela - Customers

##Configuração do caminho

In [0]:
customers_path = "/Volumes/workspace/default/logistics_operations_raw/customers.csv"

##Leitura do Arquivo

In [0]:
# Lê o arquivo CSV e cria o Dataframe
df_customers = (
    spark.read
    .option("header", True)
    .option("InferSchema", True)
    .csv(customers_path)
)

##Visualização

In [0]:
display(df_customers.limit(10))

##Tipos das colunas

In [0]:
df_customers.printSchema()

##Validar a chave customer_id

In [0]:
total_linhas_customers = df_customers.count()

total_customers_unicos = (
    df_customers
    .select("customer_id")
    .distinct()
    .count()
)

customer_id_nulos = (
    df_customers
    .filter(F.col("customer_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_customers}")
print(f"Customers únicos: {total_customers_unicos}")
print(f"Customer IDs nulos: {customer_id_nulos}")

##Verificar nulos em todas as colunas

In [0]:
display(
    df_customers.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_customers.columns
    ])
)

##Conferir valores categóricos

In [0]:
display(
    df_customers
    .groupBy("customer_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_customers
    .groupBy("primary_freight_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_customers
    .groupBy("account_status")
    .count()
    .orderBy(F.desc("count"))
)

##Validar regras numéricas e de data

In [0]:
display(
    df_customers.select(
        F.sum(
            F.when(F.col("credit_terms_days") <= 0, 1).otherwise(0)
        ).alias("prazo_credito_nao_positivo"),

        F.sum(
            F.when(F.col("annual_revenue_potential") < 0, 1).otherwise(0)
        ).alias("receita_potencial_negativa"),

        F.sum(
            F.when(F.col("contract_start_date") > F.current_date(), 1).otherwise(0)
        ).alias("contratos_com_data_futura")
    )
)

##Validar relacionamento com loads

In [0]:
customers_nao_encontrados = (
    df_loads
    .join(
        df_customers.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print(
    f"Loads com customer_id não encontrado em customers: "
    f"{customers_nao_encontrados.count()}"
)

In [0]:
customers_sem_load = (
    df_customers
    .join(
        df_loads.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
)

print(
    f"Customers sem carga correspondente: "
    f"{customers_sem_load.count()}"
)

##Grava tabela Delta

In [0]:
(
    df_customers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.customers")
)

In [0]:
spark.table("workspace.bronze_logistics.customers").count()

##Ingestão da tabela - Routes

##Configuração do caminho

In [0]:
routes_path = "/Volumes/workspace/default/logistics_operations_raw/routes.csv"

##Leitura do Arquivo

In [0]:
# Lê o arquivo CSV e cria o Dataframe
df_routes = (
    spark.read
    .option("header", True)
    .option("InferSchema", True)
    .csv(routes_path)
)

##Visualização

In [0]:
display(df_routes.limit(10))

##Tipos das colunas

In [0]:
df_routes.printSchema()

##Validar a chave route_id

In [0]:
total_linhas_routes = df_routes.count()

total_routes_unicos = (
    df_routes
    .select("route_id")
    .distinct()
    .count()
)

routes_id_nulos = (
    df_routes
    .filter(F.col("route_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_routes}")
print(f"Routes únicos: {total_routes_unicos}")
print(f"Routes IDs nulos: {routes_id_nulos}")

##Verificar nulos

In [0]:
display(
    df_routes.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_routes.columns
    ])
)

##Conferir valores categóricos

In [0]:
display(
    df_routes
    .groupBy("origin_state")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_routes
    .groupBy("destination_state")
    .count()
    .orderBy(F.desc("count"))
)

##Pares origem de destino

In [0]:
display(
    df_routes
    .groupBy(
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state"
    )
    .count()
    .orderBy(F.desc("count"))
)

##Verificar rotas duplicadas pelo trajeto

In [0]:
display(
    df_routes
    .groupBy(
        "origin_city",
        "origin_state",
        "destination_city",
        "destination_state"
    )
    .count()
    .filter(F.col("count") > 1)
)

##Validar regras numéricas

In [0]:
display(
    df_routes.select(
        F.sum(
            F.when(F.col("typical_distance_miles") <= 0, 1).otherwise(0)
        ).alias("distancia_nao_positiva"),

        F.sum(
            F.when(F.col("base_rate_per_mile") <= 0, 1).otherwise(0)
        ).alias("tarifa_base_nao_positiva"),

        F.sum(
            F.when(F.col("fuel_surcharge_rate") < 0, 1).otherwise(0)
        ).alias("sobretaxa_combustivel_negativa"),

        F.sum(
            F.when(F.col("typical_transit_days") <= 0, 1).otherwise(0)
        ).alias("prazo_transito_nao_positivo")
    )
)

##Validar relacionamento com loads

In [0]:
routes_nao_encontradas = (
    df_loads
    .join(
        df_routes.select("route_id"),
        on="route_id",
        how="left_anti"
    )
)

print(
    f"Loads com route_id não encontrado em routes: "
    f"{routes_nao_encontradas.count()}"
)

In [0]:
routes_sem_load = (
    df_routes
    .join(
        df_loads.select("route_id").distinct(),
        on="route_id",
        how="left_anti"
    )
)

print(
    f"Routes sem carga correspondente: "
    f"{routes_sem_load.count()}"
)

##Gravar a tabela Delta

In [0]:
(
    df_routes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.routes")
)

In [0]:
spark.table("workspace.bronze_logistics.routes").count()

##Ingestão da tabela - Drivers

##Configuração do Caminho

In [0]:
drivers_path = "/Volumes/workspace/default/logistics_operations_raw/drivers.csv"

##Leitura do Arquivo

In [0]:
# Lê o arquivo CSV e cria o Dataframe
df_drivers = (
    spark.read
    .option("header", True)
    .option("InferSchema", True)
    .csv(drivers_path)
)

##Visualização

In [0]:
display(df_drivers.limit(10))

##Tipos das colunas

In [0]:
df_drivers.printSchema()

##Validar a chave driver_id

In [0]:
total_linhas_drivers = df_drivers.count()

total_drivers_unicos = (
    df_drivers
    .select("driver_id")
    .distinct()
    .count()
)

drivers_id_nulos = (
    df_drivers
    .filter(F.col("driver_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_drivers}")
print(f"Drivers únicos: {total_drivers_unicos}")
print(f"Drivers IDs nulos: {drivers_id_nulos}")

##Verificar nulos

In [0]:
display(
    df_drivers.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_drivers.columns
    ])
)

##Conferir valores categóricos

In [0]:
display(
   df_drivers
    .groupBy("first_name")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
     df_drivers
    .groupBy("last_name")
    .count()
    .orderBy(F.desc("count"))
)

##Pares origem de destino

In [0]:
display(
    df_drivers
    .groupBy(
        "first_name",
        "last_name",
        "hire_date",
        "termination_date"
    )
    .count()
    .orderBy(F.desc("count"))
)

##Verificar drivers duplicados

In [0]:
display(
    df_drivers
    .groupBy(
        "first_name",
        "last_name",
        "hire_date",
        "termination_date"
    )
    .count()
    .filter(F.col("count") > 1)
)

##Validar relacionamento com drivers

In [0]:
drivers_nao_encontrados = (
    df_trips
    .filter(F.col("driver_id").isNotNull())
    .join(
        df_drivers.select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
)

print(
    f"Trips com driver_id não encontrado em drivers: "
    f"{drivers_nao_encontrados.count()}"
)

In [0]:
drivers_sem_trip = (
    df_drivers
    .join(
        df_trips
        .filter(F.col("driver_id").isNotNull())
        .select("driver_id")
        .distinct(),
        on="driver_id",
        how="left_anti"
    )
)

print(
    f"Drivers sem viagem correspondente: "
    f"{drivers_sem_trip.count()}"
)

##Verificar os motoristas sem viagem

In [0]:
display(
    drivers_sem_trip
    .select(
        "driver_id",
        "first_name",
        "last_name",
        "hire_date",
        "termination_date",
        "employment_status",
        "years_experience"
    )
    .limit(30)
)

##Por status

In [0]:
display(
    drivers_sem_trip
    .groupBy("employment_status")
    .count()
    .orderBy(F.desc("count"))
)

##Gravar a tabela Delta


In [0]:
(
    df_drivers.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.drivers")
)

In [0]:
spark.table("workspace.bronze_logistics.drivers").count()

##Ingestão da tabela - Trucks

##Configuração do Caminho

In [0]:
trucks_path = "/Volumes/workspace/default/logistics_operations_raw/trucks.csv"

##Leitura do arquivo

In [0]:
df_trucks = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(trucks_path)
)

##Visualização

In [0]:
display(df_trucks.limit(10))

##Tipos de colunas

In [0]:
df_trucks.printSchema()

##Validar a chave truck_id

In [0]:
total_linhas_trucks = df_trucks.count()

total_trucks_unicos = (
    df_trucks
    .select("truck_id")
    .distinct()
    .count()
)

truck_id_nulos = (
    df_trucks
    .filter(F.col("truck_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_trucks}")
print(f"Trucks únicos: {total_trucks_unicos}")
print(f"Truck IDs nulos: {truck_id_nulos}")

##Verificar nulos em todas as colunas

In [0]:
display(
    df_trucks.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_trucks.columns
    ])
)

##Verificar duplicidade de unit_number e vin

In [0]:
display(
    df_trucks
    .groupBy("unit_number")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    df_trucks
    .groupBy("vin")
    .count()
    .filter(F.col("count") > 1)
)

##Conferir categorias

In [0]:
display(
    df_trucks
    .groupBy("make")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_trucks
    .groupBy("fuel_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
display(
    df_trucks
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

##Validar regras numéricas e datas

In [0]:
display(
    df_trucks.select(
        F.sum(
            F.when(F.col("model_year") <= 0, 1).otherwise(0)
        ).alias("ano_modelo_invalido"),

        F.sum(
            F.when(F.col("acquisition_mileage") < 0, 1).otherwise(0)
        ).alias("quilometragem_aquisicao_negativa"),

        F.sum(
            F.when(F.col("tank_capacity_gallons") <= 0, 1).otherwise(0)
        ).alias("capacidade_tanque_nao_positiva"),

        F.sum(
            F.when(F.col("acquisition_date") > F.current_date(), 1).otherwise(0)
        ).alias("data_aquisicao_futura")
    )
)

In [0]:
display(
    df_trucks.select(
        F.sum(
            F.when(
                F.col("model_year") > F.year("acquisition_date"),
                1
            ).otherwise(0)
        ).alias("modelo_posterior_ao_ano_de_aquisicao")
    )
)

##Validar relacionamento com trips

In [0]:
trucks_nao_encontrados = (
    df_trips
    .filter(F.col("truck_id").isNotNull())
    .join(
        df_trucks.select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
)

print(
    f"Trips com truck_id não encontrado em trucks: "
    f"{trucks_nao_encontrados.count()}"
)

In [0]:
trucks_sem_trip = (
    df_trucks
    .join(
        df_trips
        .filter(F.col("truck_id").isNotNull())
        .select("truck_id")
        .distinct(),
        on="truck_id",
        how="left_anti"
    )
)

print(
    f"Trucks sem viagem correspondente: "
    f"{trucks_sem_trip.count()}"
)

In [0]:
display(
    trucks_sem_trip
    .groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

##Gravar como Delta

In [0]:
(
    df_trucks.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.trucks")
)

In [0]:
spark.table("workspace.bronze_logistics.trucks").count()

##Ingestão da tabela - delivery_events

##Caminho e leitura

In [0]:
delivery_events_path = "/Volumes/workspace/default/logistics_operations_raw/delivery_events.csv"

df_delivery_events = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(delivery_events_path)
)

##Visualização e schema

In [0]:
display(df_delivery_events.limit(10))

In [0]:
df_delivery_events.printSchema()

##Validar a chave event_id

In [0]:
total_linhas_delivery_events = df_delivery_events.count()

total_eventos_unicos = (
    df_delivery_events
    .select("event_id")
    .distinct()
    .count()
)

event_id_nulos = (
    df_delivery_events
    .filter(F.col("event_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_delivery_events}")
print(f"Eventos únicos: {total_eventos_unicos}")
print(f"Event IDs nulos: {event_id_nulos}")

##Verificar nulos

In [0]:
display(
    df_delivery_events.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_delivery_events.columns
    ])
)

##Verificar duplicidade da chave

In [0]:
display(
    df_delivery_events
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)

##Conferir categorias automaticamente

In [0]:
colunas_texto_delivery = [
    campo.name
    for campo in df_delivery_events.schema.fields
    if campo.dataType.simpleString() == "string"
    and campo.name not in ["event_id", "load_id", "trip_id", "facility_id"]
]

for coluna in colunas_texto_delivery:
    quantidade_distintos = (
        df_delivery_events
        .select(coluna)
        .distinct()
        .count()
    )

    if quantidade_distintos <= 20:
        print(f"Categorias da coluna: {coluna}")

        display(
            df_delivery_events
            .groupBy(coluna)
            .count()
            .orderBy(F.desc("count"))
        )

##Validar relacionamento com trips

In [0]:
delivery_trips_nao_encontradas = (
    df_delivery_events
    .filter(F.col("trip_id").isNotNull())
    .join(
        df_trips.select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
)

print(
    f"Eventos com trip_id não encontrado em trips: "
    f"{delivery_trips_nao_encontradas.count()}"
)

##Validar relacionamento com loads

In [0]:
delivery_loads_nao_encontradas = (
    df_delivery_events
    .filter(F.col("load_id").isNotNull())
    .join(
        df_loads.select("load_id"),
        on="load_id",
        how="left_anti"
    )
)

print(
    f"Eventos com load_id não encontrado em loads: "
    f"{delivery_loads_nao_encontradas.count()}"
)

##Verificar viagens sem evento

In [0]:
trips_sem_delivery_event = (
    df_trips
    .join(
        df_delivery_events
        .select("trip_id")
        .distinct(),
        on="trip_id",
        how="left_anti"
    )
)

print(
    f"Trips sem evento de entrega correspondente: "
    f"{trips_sem_delivery_event.count()}"
)

##Gravar como Delta

In [0]:
(
    df_delivery_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.delivery_events")
)

##Validar a gravação

In [0]:
spark.table(
    "workspace.bronze_logistics.delivery_events"
).count()

##Ingestão da tabela - fuel_purchase_id

##Caminho e leitura

In [0]:
fuel_purchases_path = "/Volumes/workspace/default/logistics_operations_raw/fuel_purchases.csv"

df_fuel_purchases = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(fuel_purchases_path)
)

##Visualização e schema

In [0]:
display(df_fuel_purchases.limit(10))

In [0]:
df_fuel_purchases.printSchema()

##Validar a chave fuel_purchase_id

In [0]:
total_linhas_fuel = df_fuel_purchases.count()

total_abastecimentos_unicos = (
    df_fuel_purchases
    .select("fuel_purchase_id")
    .distinct()
    .count()
)

fuel_purchase_id_nulos = (
    df_fuel_purchases
    .filter(F.col("fuel_purchase_id").isNull())
    .count()
)

print(f"Total de linhas: {total_linhas_fuel}")
print(f"Abastecimentos únicos: {total_abastecimentos_unicos}")
print(f"Fuel purchase IDs nulos: {fuel_purchase_id_nulos}")

##Verificar nulos

In [0]:
display(
    df_fuel_purchases.select([
        F.count(
            F.when(F.col(coluna).isNull(), 1)
        ).alias(coluna)
        for coluna in df_fuel_purchases.columns
    ])
)

##Verificar duplicidade da chave

In [0]:
display(
    df_fuel_purchases
    .groupBy("fuel_purchase_id")
    .count()
    .filter(F.col("count") > 1)
)

##Conferir categorias automaticamente

In [0]:
colunas_texto_fuel = [
    campo.name
    for campo in df_fuel_purchases.schema.fields
    if campo.dataType.simpleString() == "string"
    and campo.name not in [
        "fuel_purchase_id",
        "trip_id",
        "truck_id",
        "driver_id"
    ]
]

for coluna in colunas_texto_fuel:
    quantidade_distintos = (
        df_fuel_purchases
        .select(coluna)
        .distinct()
        .count()
    )

    if quantidade_distintos <= 20:
        print(f"Categorias da coluna: {coluna}")

        display(
            df_fuel_purchases
            .groupBy(coluna)
            .count()
            .orderBy(F.desc("count"))
        )

##Validar métricas numéricas automaticamente

In [0]:
colunas_numericas_fuel = [
    campo.name
    for campo in df_fuel_purchases.schema.fields
    if campo.dataType.simpleString() in [
        "int",
        "bigint",
        "double",
        "float",
        "decimal"
    ]
]

display(
    df_fuel_purchases.select([
        F.sum(
            F.when(F.col(coluna) < 0, 1).otherwise(0)
        ).alias(f"{coluna}_negativo")
        for coluna in colunas_numericas_fuel
    ])
)

##Validar relacionamento com trips

In [0]:
fuel_trips_nao_encontradas = (
    df_fuel_purchases
    .filter(F.col("trip_id").isNotNull())
    .join(
        df_trips.select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
)

print(
    f"Abastecimentos com trip_id não encontrado em trips: "
    f"{fuel_trips_nao_encontradas.count()}"
)

##Validar relacionamento com trucks

In [0]:
fuel_trucks_nao_encontrados = (
    df_fuel_purchases
    .filter(F.col("truck_id").isNotNull())
    .join(
        df_trucks.select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
)

print(
    f"Abastecimentos com truck_id não encontrado em trucks: "
    f"{fuel_trucks_nao_encontrados.count()}"
)

##Validar relacionamento com drivers

In [0]:
fuel_drivers_nao_encontrados = (
    df_fuel_purchases
    .filter(F.col("driver_id").isNotNull())
    .join(
        df_drivers.select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
)

print(
    f"Abastecimentos com driver_id não encontrado em drivers: "
    f"{fuel_drivers_nao_encontrados.count()}"
)

##Verificar viagens sem abastecimento

In [0]:
trips_sem_abastecimento = (
    df_trips
    .join(
        df_fuel_purchases
        .select("trip_id")
        .distinct(),
        on="trip_id",
        how="left_anti"
    )
)

print(
    f"Trips sem abastecimento correspondente: "
    f"{trips_sem_abastecimento.count()}"
)

In [0]:
display(
    df_fuel_purchases.select(
        F.sum(
            F.when(F.col("gallons") <= 0, 1).otherwise(0)
        ).alias("gallons_nao_positivo"),

        F.sum(
            F.when(F.col("price_per_gallon") <= 0, 1).otherwise(0)
        ).alias("preco_galao_nao_positivo"),

        F.sum(
            F.when(F.col("total_cost") <= 0, 1).otherwise(0)
        ).alias("custo_total_nao_positivo")
    )
)

In [0]:
display(
    df_fuel_purchases.select(
        F.sum(
            F.when(
                F.abs(
                    F.col("total_cost") -
                    (F.col("gallons") * F.col("price_per_gallon"))
                ) > 0.05,
                1
            ).otherwise(0)
        ).alias("custos_totais_inconsistentes")
    )
)

##Gravar como Delta

In [0]:
(
    df_fuel_purchases.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze_logistics.fuel_purchases")
)

In [0]:
spark.table(
    "workspace.bronze_logistics.fuel_purchases"
).count()